# Train Matrix Factorization with WOA
This notebook focuses on optimizing Matrix Factorization hyperparameters using the Whale Optimization Algorithm.

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import os
import sys
from sklearn.model_selection import train_test_split

# Set up paths relative to this notebook
NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR = os.path.dirname(NOTEBOOK_DIR)
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'ml-1m', 'ratings.dat')
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
RESULTS_PATH = os.path.join(PROJECT_DIR, 'models', 'woa_results.json')

sys.path.append(SRC_DIR)

from mf import MatrixFactorization as MF
from metrics import rmse
from woa import WhaleOptimizationAlgorithm

In [ ]:
print(f"Loading data from: {DATA_PATH}")
ratings = pd.read_csv(
    DATA_PATH,
    sep="::",
    engine="python",
    names=["user","item","rating","timestamp"]
)

ratings = ratings[["user","item","rating"]]

user_map = {u:i for i,u in enumerate(ratings.user.unique())}
item_map = {i:j for j,i in enumerate(ratings.item.unique())}

ratings["user"] = ratings.user.map(user_map)
ratings["item"] = ratings.item.map(item_map)

n_users = len(user_map)
n_items = len(item_map)

train,test = train_test_split(
    ratings.values,
    test_size=0.2,
    random_state=42
)

print("users:", n_users)
print("items:", n_items)

In [ ]:
def fitness_function(params):
    lr, reg, k, epochs = params
    
    lr = np.clip(lr, 0.001, 0.02)
    reg = np.clip(reg, 0.01, 0.2)
    k = int(np.clip(k, 20, 80))
    epochs = int(np.clip(epochs, 5, 20)) 
    
    model = MF(n_users, n_items, k, lr, reg, epochs)
    model.train(train)
    
    score = rmse(model, test)
    
    if np.isnan(score) or np.isinf(score):
        score = 999
        
    return score

In [ ]:
print("Starting WOA Optimization...")

lb = [0.002, 0.02, 20, 5]
ub = [0.02, 0.2, 80, 20]

woa = WhaleOptimizationAlgorithm(
    fitness_func=fitness_function,
    n_whales=15,
    n_iter=15,
    lower_bound=lb,
    upper_bound=ub
)

start = time.time()
best_params, best_rmse, history = woa.optimize()
elapsed = time.time() - start

print(f"\nBest Params: LR={best_params[0]:.4f}, Reg={best_params[1]:.4f}, K={int(best_params[2])}, Epochs={int(best_params[3])}")
print(f"Best RMSE: {best_rmse:.4f}")
print(f"Optimization Time: {elapsed:.2f}s")

In [ ]:
result = {
    "lr": float(best_params[0]),
    "reg": float(best_params[1]),
    "k": int(best_params[2]),
    "epochs": int(best_params[3]),
    "rmse": float(best_rmse),
    "history": history,
    "time": elapsed
}

print(f"Saving results to: {RESULTS_PATH}")
with open(RESULTS_PATH, "w") as f:
    json.dump(result, f, indent=4)